# Spatially-Dependent Multiple Comparison Correction (SpatialFDR)

This notebook demonstrates the use of the `SpatialFDR` class in `esda`. 

When evaluating local spatial statistics (such as Local Moran's $I$), we run into the **multiplicity problem**: conducting hundreds or thousands of simultaneous hypothesis tests inflates our Type I error rate. Standard corrections like Benjamini-Hochberg (BH) assume independent tests, while corrections like Benjamini-Yekutieli (BY) are overly conservative, destroying statistical power.

`SpatialFDR` implements **Giuseppe Arbia's Spatially-Dependent BH procedure**. It uses an analytical Approximate Profile Likelihood Estimator (APLE) to measure the global spatial dependency ($\rho$) of the underlying spatial field ($y$). It then derives an **effective sample size ($n_{\text{eff}}$)** to expand the FDR critical thresholds, structurally accounting for spatial redundancy without sacrificing power.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import scipy.sparse as sparse
import scipy.sparse.linalg as spla
import libpysal
from esda.moran import Moran_Local
from esda.fdr import SpatialFDR  # Assuming package path layout



## 1. Simulate a Spatially Autocorrelated Field

We will construct a 30x30 regular grid lattice using `libpysal` ($n = 900$) and simulate a Simultaneous Autoregressive (SAR) error process with a high true spatial parameter ($\rho = 0.65$).

In [ ]:
# Set seed for reproducible synthetic data generation
np.random.seed(42)

# 1. Build a 30x30 lattice graph
l = 30
w = libpysal.weights.util.lat2W(l, l)
graph = libpysal.graph.Graph.from_W(w)
n = graph.n

# 2. Extract row-standardized sparse weights matrix
W_sparse = graph.transform('R').sparse
I_sparse = sparse.eye_array(n, format='csr')
I_sparse.setdiag(1) # Efficient sparse Identity matrix

# 3. Simulate SAR Process: y = (I - rho * W)^(-1) * epsilon
true_rho = 0.5
epsilon = np.random.normal(0, 1, size=n)
A = I_sparse - true_rho * W_sparse
y_spatial = spla.spsolve(A, epsilon)

print(f"Nominal Dataset Size (n): {n}")
print(f"Simulated Data Mean:      {y_spatial.mean():.4f}")
print(f"Simulated Data Variance:  {y_spatial.var():.4f}")

## 2. Compute Local Spatial Statistics (Local Moran's $I$)

Next, we run standard local spatial statistics via `esda.moran.Moran_Local`. We extract the raw, unadjusted simulation-based $p$-values (`p_sim`).

In [ ]:
lm = Moran_Local(y_spatial, graph, transformation='r', permutations=999)
raw_pvalues = lm.p_sim

print(f"Calculated {len(raw_pvalues)} local p-values.")

In [ ]:
(lm.p_sim < 0.05).sum()

## 3. Execute Spatially-Dependent FDR Correction

Now we pass both our unadjusted local `raw_pvalues`, the raw spatial process layer `y_spatial`, and our connectivity `graph` into `SpatialFDR`.

In [ ]:
# Fit Arbia's Spatial FDR correction
fdr = SpatialFDR(pvalues=raw_pvalues, y=y_spatial, graph=graph, alpha=0.05, transform='R')

print("--- Correction Parameters ---")
print(f"Estimated Spatial Rho (via APLE): {fdr.rho:.4f}")
print(f"Calculated Effective Sample Size: {fdr.n_eff:.2f} (Holds {fdr.n_eff / n * 100:.1f}% of total independent info)")
print(f"Total Local Discoveries Retained: {fdr.reject.sum()} / {n}")

In [ ]:
lm.reject = (lm.p_sim <= 0.05)
lm.reject.sum()


In [ ]:
fdr.n_eff

In [ ]:
# Calculate Benjamini-Yekutieli (BY) denominator modifier for comparison
harmonic_sum = np.sum(1.0 / np.arange(1, n + 1))
by_denominator = n * harmonic_sum

# Sort p-values for ranking visualization
sorted_p = np.sort(raw_pvalues)
ranks = np.arange(1, n + 1)

# Generate thresholds
bh_thresh = (ranks / n) * 0.05
by_thresh = (ranks / by_denominator) * 0.05

# Raw Arbia linear threshold sequence
arbia_thresh_raw = (ranks / fdr.n_eff) * 0.05
# CRITICAL FIX: Cap the threshold sequence at alpha (0.05) to mirror the _fit method
arbia_thresh = np.minimum(arbia_thresh_raw, 0.05)

# Plotting the threshold curves (Zoomed in to the first 250 ranks to see the ceiling hit)
plt.figure(figsize=(10, 6))
plt.plot(ranks[:250], sorted_p[:250], label='Observed Sorted p-values', color='black', linewidth=2)
plt.plot(ranks[:250], bh_thresh[:250], '--', label='Standard BH Threshold (Naive)', color='blue')
plt.plot(ranks[:250], arbia_thresh[:250], label="Arbia Spatial BH Threshold (n_eff capped)", color='green', linewidth=2)
plt.plot(ranks[:250], by_thresh[:250], ':', label='BY Threshold (Arbitrary Dependency)', color='red')

# Visual marker for the alpha ceiling
plt.axhline(y=0.05, color='gray', linestyle=':', alpha=0.6, label='Alpha Budget Ceiling (0.05)')

plt.title("Multiple Comparison Threshold Comparison (First 250 Ranks)", fontsize=14)
plt.xlabel("Rank (k)", fontsize=12)
plt.ylabel("p-value / Threshold", fontsize=12)
plt.ylim(-0.01, 0.07)  # Tighten y-axis to clearly inspect the 0.05 boundary crossing
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 1. Compute standard BH discoveries using the properly sorted p-values
bh_significant_sorted = sorted_p <= bh_thresh
max_k_bh = np.max(ranks[bh_significant_sorted]) if np.any(bh_significant_sorted) else 0

# 2. Compute BY discoveries using the properly sorted p-values
by_significant_sorted = sorted_p <= by_thresh
max_k_by = np.max(ranks[by_significant_sorted]) if np.any(by_significant_sorted) else 0

# 3. Arbia discoveries are already stored in our SpatialFDR instance
max_k_arbia = fdr.reject.sum()

# 4. Uncorrected base significance (naive alpha = 0.05)
uncorrected_discoveries = np.sum(raw_pvalues <= 0.05)

# Build a summary table for the User Guide documentation
summary_data = {
    "Correction Method": [
        "No Correction (Naive Local Alpha)",
        "Arbia Spatially-Dependent BH",
        "Standard Benjamini-Hochberg",
        "Benjamini-Yekutieli (BY)"
    ],
    "Effective Sample Size (n_eff Used)": [
        f"{n}", 
        f"{fdr.n_eff:.2f}", 
        f"{n}", 
        f"{by_denominator:.2f}"
    ],
    "Local Hotspots Detected": [
        uncorrected_discoveries,
        max_k_arbia,
        max_k_bh,
        max_k_by
    ],
    "Statistical Power Remaining": [
        "100% (High False Positive Risk)",
        f"{max_k_arbia / uncorrected_discoveries * 100:.1f}%",
        f"{max_k_bh / uncorrected_discoveries * 100:.1f}%",
        f"{max_k_by / uncorrected_discoveries * 100:.1f}%"
    ]
}

df_summary = pd.DataFrame(summary_data)
print("=== MULTIPLE COMPARISON PERFORMANCE METRICS ===")
print(df_summary.to_string(index=False))